# Lab 2.1 – Hybrid Store Benchmark

**Course:** Advanced RAG Architecture on GCP  
**Goal:** Compare an AlloyDB-style hybrid store with a Vector-Search-style pure semantic store on the Week 1 corpus and evaluation queries.

Both backends run offline. The interface is what you will reuse in the Capstone.

In [ ]:
# --- Environment bootstrap (Colab + local + Docker) ---
import os, sys
from pathlib import Path

def _in_colab() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

REPO_URL = "https://github.com/fischer3-net/accurate_secure_rag_systems.git"
LAB_DIR = "labs/02-storage"

if _in_colab():
    REPO_ROOT = Path("/content/accurate_secure_rag_systems")
    if not REPO_ROOT.exists():
        get_ipython().system(f"git clone --depth 1 {REPO_URL} {REPO_ROOT}")
    LAB = REPO_ROOT / LAB_DIR
    os.chdir(LAB)
    sys.path.insert(0, str(LAB))
    sys.path.insert(0, str(REPO_ROOT / "labs" / "01-chunking"))
    get_ipython().run_line_magic("pip", "install -q pydantic python-dotenv langchain-text-splitters langchain-core pytest pyyaml pandas")
    print("Colab ready | LAB =", LAB)
else:
    LAB = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    if str(LAB) not in sys.path:
        sys.path.insert(0, str(LAB))
    print("Local ready | LAB =", LAB)


In [ ]:
import sys
from pathlib import Path

LAB_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
WEEK1 = LAB_ROOT.parent / "01-chunking"
sys.path.insert(0, str(LAB_ROOT))
sys.path.insert(0, str(WEEK1))

from src.chunking import process_directory
from src.pgvector_store import PgVectorStore
from src.vector_search_store import VectorSearchStore
from src.benchmark import run_benchmark, load_eval_queries

print("Week1 data:", WEEK1 / "data")

In [ ]:
corpus = process_directory(WEEK1 / "data")
queries = load_eval_queries(WEEK1 / "data" / "evaluation_queries.json")
print(f"Corpus: {len(corpus)} chunks | Queries: {len(queries)}")

In [ ]:
pg = PgVectorStore()
pg.upsert(corpus)
vs = VectorSearchStore()
vs.upsert(corpus)
print("pgvector rows:", pg.count(), "| vector_search rows:", vs.count())

In [ ]:
report = run_benchmark(
    {"pgvector (hybrid)": pg, "vector_search (pure)": vs},
    queries,
    k=3,
)
print(report.summary())

In [ ]:
# Filtered query example – hybrid stores shine here
r_pg = pg.search("trust boundary", top_k=3, asset_type="trust_boundary", risk_tier="critical")
r_vs = vs.search("trust boundary", top_k=3, asset_type="trust_boundary", risk_tier="critical")
print("pgvector hits:", [(h.control_id, h.metadata.get("risk_tier")) for h in r_pg.hits])
print("vector_search hits:", [(h.control_id, h.metadata.get("risk_tier")) for h in r_vs.hits])

## Recommendation (fill in for submission)

- Which backend produced the higher hit-rate@3 on the unfiltered query set?
- For *filtered* queries (asset_type / risk_tier), which interface felt more natural?
- For the Capstone interactive DFD reviewer, which store would you adopt first and why?

_Write 4–8 sentences here._